# **Phase 4 — Train RandomForestClassifier Prediction Model**
- Load dataset
- Select features
- Split data
- Train model
- Evaluate performance
- Save mdodel

### Create DuckDB Connection

In [11]:
import os
import duckdb

DB_FOLDER = "database"
DB_PATH = os.path.join(DB_FOLDER, "cse_data.db")

con = duckdb.connect(database=DB_PATH)

### Load Dataset

In [3]:
import pandas as pd
import duckdb
import os

pd.set_option("display.max_columns", None)

stock_features_labeled_10d_df = con.execute("""
    SELECT *
    FROM stocks_features_labeled_10d_table
""").fetch_df()

print(stock_features_labeled_10d_df.shape)
print(stock_features_labeled_10d_df.head())
print(stock_features_labeled_10d_df.columns)

IOException: IO Error: Cannot open file "c:\users\lapmart\jupyter ai projects\cse investment assistence with xgboost\database\cse_data.db": The process cannot access the file because it is being used by another process.

File is already open in 
C:\Users\Lapmart\anaconda3\python.exe (PID 15628)

### Select Features + Target

In [4]:
features = [
    "ret_5",
    "ret_10",
    "volume_ratio",
    "volume_z",
    "range_position",
    "std_close_10",
    "liquidity_rank"
]

X = features_labeled_10d_df[features]
y = features_labeled_10d_df["target_buy_10d"]

NameError: name 'features_labeled_10d_df' is not defined

### Train/Test Split

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False   # IMPORTANT for time-series data
)

### Check nulls or invalid values

In [26]:
import numpy as np

print("infinites :\n-----------\n", np.isinf(X_train).sum())
print("\nNans :\n------\n", np.isnan(X_train).sum())
print("\nis finite all ? \n--------------\n", np.isfinite(X_train).all())  # should return True for all columns

infinites :
-----------
 ret_5             0
ret_10            0
volume_ratio      0
volume_z          0
range_position    0
std_close_10      0
liquidity_rank    0
dtype: int64

Nans :
------
 ret_5             0
ret_10            0
volume_ratio      0
volume_z          0
range_position    0
std_close_10      0
liquidity_rank    0
dtype: int64

is finite all ? 
--------------
 ret_5             True
ret_10            True
volume_ratio      True
volume_z          True
range_position    True
std_close_10      True
liquidity_rank    True
dtype: bool


In [27]:
print(y_train.value_counts())

target_buy_10d
0    7992
1    1210
Name: count, dtype: int64


### Train Model

In [28]:
from sklearn.ensemble import RandomForestClassifier

model_random_forest = RandomForestClassifier(
    n_estimators = 200,     # Number of trees
    max_depth = None,          # Maximum depth of each tree
    random_state = 42,      # reproducible results
    n_jobs = -1,            # use all CPU cores (faster)
    min_samples_leaf=1,
    min_samples_split=2,
    class_weight="balanced" # Adjust weights inversely proportional to class frequencies (helps with imbalanced datasets)
)

model_random_forest.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

### Evaluate Model
- **Accuracy**
   - overall correctness of model
   - Accuracy = Correct Predictions / Total predictions
- **Precision**
   - When model predicts this class, how often is it correct ?
   - Precision = True Positives / Predicted positives
   - Class 1 precision = 0.X --> Only X% of predicted BUY signals were actually correct
- **Recall**
  - Out of all real items of this class, how many did model find ?
  - Recall = True Positives / Actual Positives
  - Class 1 recall = 0.X --> Model detected X% of real BUY opportunities
- **F1-Score**
   - Balance between precision and recall
   - F1 = 2 × ( Precision × Recall ) / ( Precision + Recall )
   - Class 1 f1 = 0.X --> Overall prediction quality for BUY class is moderate
- **Support**
   - support = number of true samples
- **macro avg**
   - Macro average = simple average of classes
   - Treats both classes equally
   - Eg : Macro Precision = ( Precision0 + Precision1 ) / 2
   - Used when, you want fairness across classes
- **weighted avg**
   - Weighted average = average weighted by class size
   - Weighted = ( metric0 x n0 + metric1 × n1 ) / total
   - Used when, dataset is imbalanced and you want realistic performance

#### Which metric matters most?

| Goal | Best metric |
|------|-------------|
| Avoid false trades | precision |
| Find all good trades | recall |
| Balanced system | f1-score |
| Overall correctness | accuracy |

In [29]:
from sklearn.metrics import accuracy_score, classification_report

preds = model_random_forest.predict(X_test)

print("Accuracy Random Forest :\n\n", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy Random Forest :

 0.8696219035202086
              precision    recall  f1-score   support

           0       0.88      0.99      0.93      2024
           1       0.17      0.02      0.04       277

    accuracy                           0.87      2301
   macro avg       0.53      0.50      0.48      2301
weighted avg       0.80      0.87      0.82      2301



#### Save Model

In [30]:
import joblib
import os

os.makedirs("pkls", exist_ok=True)
joblib.dump(model_random_forest, "pkls/stock_model_randomforest.pkl")

['pkls/stock_model_randomforest.pkl']

## **Phase 4.5 (Tuning)**
- We must optimize : scoring = "f1", actualy not accuracy

### 🌲 RandomForestClassifier (Model)
- A machine learning algorithm that makes predictions using many decision trees
- Instead of asking one expert, it asks many experts and takes a vote
- Single decision trees can be wrong, but Many trees together → more accurate and stable
- We don't use all varible for each tree we use sqrt of no of varible to train a particular tree
- then we get the majority vote

### 🔍 GridSearchCV (Hyperparameter tuner)
- A tool that automatically finds the best model settings
- in simple words - automatic parameter testerYou
- It  try all combinations and pick best

### 🔁 StratifiedKFold (Data splitting method)
- A smarter way to split data for validation
- Fold = data group we split into 80, 20 %, but if we have only 0 in a choosen group it dont know about 1, it split 0 group ino agian train test split no meaning
- Problem it solves
  - If your dataset is imbalanced 80% class 0, 20% class 1 --> Random splitting might produce : Fold 1 → 95% class 0, Fold 2 → 100% class 0
  - StratifiedKFold keeps same class ratio in every split > 80% class 0, 20% class 1

In [31]:
# 108 combinations
# With CV=5 →
# 108 × 5 = 540 model trainings

n_estimators:      [100, 300, 500]
max_depth:         [5, 10, 20, None]
min_samples_split: [2, 5, 10]
min_samples_leaf:  [1, 2, 4]

In [32]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators": [100, 300, 500],   # Number of trees in the forest to test -> More trees = usually better but slower
    "max_depth": [5, 10, 20, None],    # Maximum depth of each tree: small depth → simple model / large depth → complex model / None → grow fully until pure
    "min_samples_split": [2, 5, 10],   # Minimum samples needed to split a node: smaller → more splits → more complex model / larger → fewer splits → simpler model
    "min_samples_leaf": [1, 2, 4]      # Minimum samples required in each leaf node: prevents tiny leaves / helps reduce overfitting
}

# random_state=42 → reproducible results
# n_jobs=-1 → use all CPU cores (faster)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# Creates cross-validation splitter
# n_splits=5 → 5-fold validation
# Stratified → keeps class balance in each fold
# cv is just a configuration object, It is just an instruction object that knows how to split data, But it does NOT split until .fit() is called
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Creates a Grid Search object that will test all parameter combinations
grid = GridSearchCV(
    estimator=rf,            # Model to tune → RandomForest
    param_grid=param_grid,   # Search space → the combinations defined earlier
    scoring="f1",            # Metric used to judge models → F1 score >>> balances precision + recall, better than accuracy for imbalanced data
    cv=cv,                   # Use custom cross-validation splitter
    verbose=3,               # Print progress while training, 1 - no, 2, more, 3- morea detailos than 2, .. so on..
    n_jobs=-1                # Run parameter search using all CPU cores
)

# Starts training
# For each hyperparameter combination : 
#    Train model on 4 folds
#    Test on remaining fold
#    Repeat 5 times
#    Compute average F1 score
grid.fit(X_train, y_train)

# Prints the best hyperparameters found.
print("Best Params:", grid.best_params_)

# Prints best average cross-validation F1 score
print("Best Score:", grid.best_score_)

# Stores the best trained model so you can use it
best_model = grid.best_estimator_

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best Score: 0.25099180357851775


In [39]:
y_pred = best_model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.98      0.93      2024
           1       0.24      0.04      0.07       277

    accuracy                           0.87      2301
   macro avg       0.56      0.51      0.50      2301
weighted avg       0.80      0.87      0.83      2301



### Close DuckDB Connection

In [3]:
if 'con' in globals():  # Check if connection exists
    try:
        con.close()        # Close it
        print("DuckDB connection closed.")
    except Exception as e:
        print("Error closing DuckDB:", e)
    finally:
        del con             # Delete the variable from memory